# Economic Pulse — Colab Model Training

Train the Economic Pulse monthly activity forecasting pipeline on your own CSV using a Google Colab runtime. This notebook uses the repository's existing validation, as-of feature engineering, chronological backtesting, XGBoost challenger, model-card and bundle-publication code.

**Current target:** monthly positive transaction counts (initially UPI activity), not GDP. Do not interpret this model as a GDP forecast.

Run the cells from top to bottom. Your data stays in this Colab runtime unless you explicitly download artifacts or submit the CSV to the API.

In [ ]:
# 1. Clone the latest repository and install the model-training dependencies
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/MAINAKRUDRA/ECONOMIC-PULSE.git"
REPO_DIR = Path('/content/ECONOMIC-PULSE')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/ECONOMIC-PULSE
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)
print('Repository ready:', REPO_DIR)


## 2. Prepare your data

Upload one CSV for one monthly series. Required columns:

| Column | Meaning |
|---|---|
| `series_id` | Stable series identifier, e.g. `upi_monthly` |
| `period` | Reference month, first day, e.g. `2024-01-01` |
| `available_at` | Actual timestamp when that observation/vintage became available |
| `revision` | Non-negative integer vintage number |
| `volume_count` | Positive numeric count, consistent units |

If your file contains only the latest revised snapshot, do not invent historical release dates or revisions. Use the real capture date and understand that historical as-of backtesting is limited.

In [ ]:
# 3. Upload your CSV from your computer
from google.colab import files
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_names) != 1:
    raise ValueError(f'Upload exactly one CSV file. CSV files detected: {csv_names}')
CSV_PATH = Path('/content') / csv_names[0]
print('Selected:', CSV_PATH)


In [ ]:
# 4. Inspect and validate the dataset before training
import pandas as pd
from packages.ml.data import load_vintages

raw = load_vintages(CSV_PATH)
display(raw.head(10))
print('Rows:', len(raw))
print('Series:', sorted(raw['series_id'].unique().tolist()))
print('Period range:', raw['period'].min(), 'to', raw['period'].max())
print('Distinct months:', raw['period'].nunique())
print('Vintages per month (top):')
display(raw.groupby('period').size().describe().to_frame('vintages_per_period'))


In [ ]:
# 5. Select the series and record provenance
# Change SERIES_ID to exactly match one value printed in the previous cell.
SERIES_ID = sorted(raw['series_id'].unique().tolist())[0]
SOURCE_NAME = 'REPLACE WITH OFFICIAL DATA SOURCE'
RIGHTS_TAG = 'reviewed-public-use'

assert SERIES_ID in set(raw['series_id']), 'SERIES_ID is not present in the uploaded CSV'
assert not SOURCE_NAME.startswith('REPLACE'), 'Set SOURCE_NAME to the real source before training'
assert RIGHTS_TAG.strip(), 'Document the data usage/rights status'
print('Training series:', SERIES_ID)
print('Source:', SOURCE_NAME)
print('Rights tag:', RIGHTS_TAG)


## 6. Train and evaluate on Colab

This runs the repository's real pipeline locally in the Colab runtime: strict CSV validation, release-aware feature construction, flat/trend/seasonal baselines, expanding chronological backtests, XGBoost point and quantile challengers, model selection, forecast generation, hashes and model card. The XGBoost challenger is promoted only if it beats the selected baseline on the required validation sample.

In [ ]:
from packages.ml.train import run_training
from pathlib import Path
import json

OUTPUT_DIR = Path('/content/economic-pulse-colab-artifacts')
result = run_training(
    csv_path=CSV_PATH,
    out_dir=OUTPUT_DIR,
    series_id=SERIES_ID,
    source_name=SOURCE_NAME,
    rights_tag=RIGHTS_TAG,
)
print(json.dumps(result, indent=2, default=str))
print('\nArtifacts written to:', OUTPUT_DIR)


In [ ]:
# 7. Review the forecast, model selection, validation and backtest results
from IPython.display import display, Markdown

forecast_rows = json.loads((OUTPUT_DIR / 'forecast_rows.json').read_text())
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
challenger = json.loads((OUTPUT_DIR / 'challenger_validation.json').read_text())
backtest = json.loads((OUTPUT_DIR / 'backtest.json').read_text())
display(pd.DataFrame(forecast_rows))
display(Markdown((OUTPUT_DIR / 'model_card.md').read_text()))
print('Training rows:', manifest.get('training_rows'))
print('Selected model:', manifest.get('selection'))
print('Candidate status:', manifest.get('candidate_status'))
print('Challenger validation:')
print(json.dumps(challenger, indent=2))
print('Backtest:')
print(json.dumps(backtest, indent=2))


In [ ]:
# 8. Package and download the Colab training artifacts
import shutil
from google.colab import files

ARCHIVE_BASE = '/content/economic-pulse-colab-run'
archive_path = shutil.make_archive(ARCHIVE_BASE, 'zip', OUTPUT_DIR)
print('Created:', archive_path)
files.download(archive_path)


## 9. Optional: submit the same CSV to the Economic Pulse API

The API currently accepts a CSV and performs its own controlled training/publication on the backend. This is a separate server-side run; it does **not** upload the Colab-trained model binaries. Use this only after setting `ECONOMIC_PULSE_ADMIN_TOKEN` in the Render API service. Never paste the token into a notebook cell or commit it to GitHub; enter it using the hidden prompt below.

In [ ]:
# Optional backend publication (server retrains from the CSV)
import getpass, time, requests

PUBLISH_TO_API = False  # Change to True only when the Render admin token is configured.
API_BASE_URL = 'https://economic-pulse-api.onrender.com'

if PUBLISH_TO_API:
    admin_token = getpass.getpass('Economic Pulse API admin token: ')
    headers = {'Authorization': f'Bearer {admin_token}'}
    with open(CSV_PATH, 'rb') as data_file:
        response = requests.post(
            f'{API_BASE_URL}/v1/admin/train',
            headers=headers,
            data={'series_id': SERIES_ID, 'source_name': SOURCE_NAME, 'rights_tag': RIGHTS_TAG},
            files={'file': (CSV_PATH.name, data_file, 'text/csv')},
            timeout=120,
        )
    response.raise_for_status()
    job = response.json()
    print('Backend job:', job)
    job_url = API_BASE_URL + job.get('status_url', f"/v1/admin/train/{job['job_id']}")
    for attempt in range(120):
        status_response = requests.get(job_url, headers=headers, timeout=30)
        status_response.raise_for_status()
        status = status_response.json()
        print(status)
        if status.get('status') in {'completed', 'failed'}:
            break
        time.sleep(5)
    if status.get('status') != 'completed':
        raise RuntimeError(f'Backend training did not complete successfully: {status}')
    print('Published. Check:', f'{API_BASE_URL}/v1/status', f'{API_BASE_URL}/v1/model-card')
else:
    print('Skipped. Set PUBLISH_TO_API=True to run the separate backend publication step.')


## Interpretation and limitations

- A successful run means the pipeline executed; it does not automatically mean the model is accurate or production-ready.
- Check the number of validation rows, baseline MAE, XGBoost MAE, holdout metrics and interval calibration before trusting the forecast.
- If XGBoost is not promoted, the pipeline intentionally keeps the better baseline.
- A single latest-revised snapshot cannot support genuine historical release-aware backtesting. Do not fabricate vintage timestamps.
- This initial target forecasts monthly transaction counts. It is not a GDP, inflation, employment or broad economic-growth forecast.
- Colab files are temporary. Download the ZIP before closing or disconnecting the runtime.